In [1]:
import pandas as pd
import numpy as np
import warnings
import seaborn as sns

In [2]:
warnings.filterwarnings('ignore')
sns.set_style('whitegrid')
sns.set_style("ticks")
odx = pd.IndexSlice
pd.set_option('display.max_columns', 100)
pd.set_option('display.max_rows', 100)
pd.set_option('display.float_format', lambda x: '%.2f' % x)

In [3]:
def read_url(link):
    """ Creates a pandas DataFrame from data online
    - Parameters:
        - link: link to the zipped data
    - Returns:
    """
    import io
    import requests
    import pandas as pd

    # Define URL and extract information
    response = requests.get(link)
    content = response.content
    # Convert into a Pandas DataFrame
    df = pd.read_csv(io.BytesIO(content), sep=',', compression='gzip')

    return df

In [48]:
listings = read_url('https://data.insideairbnb.com/mexico/df/mexico-city/2025-06-25/data/listings.csv.gz')
listings['price'] = listings['price'].replace('[\$,]', '', regex=True).astype(float)
print(listings.shape)

(26401, 79)


In [5]:
listings['amenities']

0        ["Kitchen", "Resort access", "Hot water", "Cou...
1        ["Free street parking", "Free parking on premi...
2        ["Dining table", "Hot water", "Hangers", "Esse...
3        ["Hot water", "TV with standard cable", "Hange...
4        ["Varies conditioner", "Dining table", "Free s...
                               ...                        
26396    ["Self check-in", "Carbon monoxide alarm", "Wa...
26397    ["Air conditioning", "Kitchen", "Smoke alarm",...
26398    ["Carbon monoxide alarm", "First aid kit", "Ki...
26399    ["Dining table", "Free street parking", "Pool ...
26400    ["Dining table", "Free parking on premises", "...
Name: amenities, Length: 26401, dtype: object

In [6]:
listings['description']

0        Dentro de Villa un estudio de arte con futon, ...
1        A new concept of hosting in mexico through a b...
2        This great apartment is located in one of the ...
3        Comfortably furnished, sunny, 2 bedroom apt., ...
4        COYOACAN designer studio quiet & safe! well eq...
                               ...                        
26396    New 60-meter luxury apartment with two spaciou...
26397    Experience elevated business travel in our bea...
26398    Apartment in the heart of Mexico City, 2 minut...
26399    Stay in a chic loft inside a restored historic...
26400    Enjoy a stylish stay in this spacious 3BR apar...
Name: description, Length: 26401, dtype: object

In [7]:
listings['name']

0                                             Villa Dante
1                                            Condesa Haus
2                    Great space in historical San Rafael
3                       2 bedroom apt. deco bldg, Condesa
4        Beautiful light Studio Coyoacan- full equipped !
                               ...                       
26396    Central 68 Arena CDMX Aduana Pantaco Ind Vallejo
26397                          CDMX | Business Class Flat
26398                     Corazón CDMX Roma norte/Reforma
26399                          Chic Loft + Lap Pool & Gym
26400                 Stylish 3 BR & Terrace near Reforma
Name: name, Length: 26401, dtype: object

In [8]:
listings['amenities_parsed'] = listings['amenities'].apply(lambda x: str(x).strip('{}').replace('"', '').replace('[', '').replace(']', '').split(', '))

In [9]:
listings['amenities_parsed']

0        [Kitchen, Resort access, Hot water, Courtyard ...
1        [Free street parking, Free parking on premises...
2        [Dining table, Hot water, Hangers, Essentials,...
3        [Hot water, TV with standard cable, Hangers, E...
4        [Varies conditioner, Dining table, Free street...
                               ...                        
26396    [Self check-in, Carbon monoxide alarm, Washer,...
26397    [Air conditioning, Kitchen, Smoke alarm, Exter...
26398    [Carbon monoxide alarm, First aid kit, Kitchen...
26399    [Dining table, Free street parking, Pool table...
26400    [Dining table, Free parking on premises, Condi...
Name: amenities_parsed, Length: 26401, dtype: object

In [47]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import nltk

# Descargar stopwords de NLTK (solo la primera vez)
nltk.download('stopwords')
from nltk.corpus import stopwords

spanish_stopwords = set(stopwords.words('spanish'))
english_stopwords = set(ENGLISH_STOP_WORDS)

combined_stopwords = list(english_stopwords.union(spanish_stopwords))

# campo de contenido (amenities + description)
def normalize_text(text):
    if isinstance(text, str):
        return text.lower()
    return ""

listings['content'] = (
    listings['amenities_parsed'].apply(lambda x: ' '.join(x)) + ' ' +
    listings['description'].fillna('').apply(normalize_text)
)

# TF-IDF para inglés + español
tfidf_vectorizer = TfidfVectorizer(
    max_features=500,
    stop_words=combined_stopwords,
)

tfidf_embeddings = tfidf_vectorizer.fit_transform(listings['content'])

# Matriz de similitud
similarity_matrix = cosine_similarity(tfidf_embeddings)


# Función para obtener similares 
def recommend_from_description(user_description, top_n=5):
    """
    Recommend listings based on a user-provided natural language description.
    """
    # Normalize text
    user_description = user_description.lower()

    # Convert to vector with the SAME tf-idf model used for training
    user_vec = tfidf_vectorizer.transform([user_description])

    # Compute similarity vs ALL listings
    similarities = cosine_similarity(user_vec, tfidf_embeddings).flatten()

    # Get top N similar listing indices
    similar_indices = np.argsort(similarities)[::-1][:top_n]

    # Build result dataframe
    results = listings.iloc[similar_indices][[
        'id', 'name', 'price', 'amenities_parsed'
    ]].copy()

    results['similarity_score'] = similarities[similar_indices]

    return results

# Prueba 
user_query = "Quiero un departamento moderno cerca de la playa, con wifi y alberca."
print(recommend_from_description(user_query, top_n=5))

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/gblasd/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


                        id                                            name  \
7582              45988882             Rento cuarto en Villa Centroamérica   
6671              42788505                                  Hotel Roma 191   
20516  1184051752569881475                            Acogedora Habitación   
20529  1184107079176155524                            Habitación Acogedora   
7454              45741122  Recámara privada, entre el metro Taxqueña y CU   

         price                                   amenities_parsed  \
7582   $212.00                         [Wifi, Kitchen, Hot water]   
6671       NaN                       [TV, Wifi, Essentials, Iron]   
20516  $441.00                        [TV, Washer, Wifi, Kitchen]   
20529  $461.00                        [TV, Washer, Wifi, Kitchen]   
7454       NaN  [Hangers, Essentials, Lock on bedroom door, Pr...   

       similarity_score  
7582               0.45  
6671               0.40  
20516              0.39  
20529       

In [ ]:
from sklearn.feature_extraction.text import TfidfVectorizer, ENGLISH_STOP_WORDS
from sklearn.metrics.pairwise import cosine_similarity
import numpy as np
import nltk

# Descargar stopwords de NLTK (solo la primera vez)
nltk.download('stopwords')
from nltk.corpus import stopwords

spanish_stopwords = set(stopwords.words('spanish'))
english_stopwords = set(ENGLISH_STOP_WORDS)

combined_stopwords = list(english_stopwords.union(spanish_stopwords))

# campo de contenido (amenities + description)
def normalize_text(text):
    if isinstance(text, str):
        return text.lower()
    return ""

listings['content'] = (
    listings['amenities_parsed'].apply(lambda x: ' '.join(x)) + ' ' +
    listings['description'].fillna('').apply(normalize_text)
)

# TF-IDF para inglés + español
tfidf_vectorizer = TfidfVectorizer(
    max_features=500,
    stop_words=combined_stopwords,
)

tfidf_embeddings = tfidf_vectorizer.fit_transform(listings['content'])

# Matriz de similitud
similarity_matrix = cosine_similarity(tfidf_embeddings)


# Función para obtener similares 
def recommend_from_description(user_description, top_n=5):
    """
    Recommend listings based on a user-provided natural language description.
    """
    # Normalize text
    user_description = user_description.lower()

    # Convert to vector with the SAME tf-idf model used for training
    user_vec = tfidf_vectorizer.transform([user_description])

    # Compute similarity vs ALL listings
    similarities = cosine_similarity(user_vec, tfidf_embeddings).flatten()

    # Get top N similar listing indices
    similar_indices = np.argsort(similarities)[::-1][:top_n]

    # Build result dataframe
    results = listings.iloc[similar_indices][[
        'id', 'name', 'price', 'amenities_parsed'
    ]].copy()

    results['similarity_score'] = similarities[similar_indices]

    return results

# Prueba 
user_query = "Quiero un departamento moderno cerca de la playa, con wifi y alberca."
print(recommend_from_description(user_query, top_n=5))

[nltk_data] Downloading package stopwords to
[nltk_data]     /Users/gblasd/nltk_data...
[nltk_data]   Package stopwords is already up-to-date!


                        id                                            name  \
7582              45988882             Rento cuarto en Villa Centroamérica   
6671              42788505                                  Hotel Roma 191   
20516  1184051752569881475                            Acogedora Habitación   
20529  1184107079176155524                            Habitación Acogedora   
7454              45741122  Recámara privada, entre el metro Taxqueña y CU   

         price                                   amenities_parsed  \
7582   $212.00                         [Wifi, Kitchen, Hot water]   
6671       NaN                       [TV, Wifi, Essentials, Iron]   
20516  $441.00                        [TV, Washer, Wifi, Kitchen]   
20529  $461.00                        [TV, Washer, Wifi, Kitchen]   
7454       NaN  [Hangers, Essentials, Lock on bedroom door, Pr...   

       similarity_score  
7582               0.45  
6671               0.40  
20516              0.39  
20529       

In [28]:
# Add values of amenities_parsed in the list if the value not exists
amenities_list = []
for amenities in listings['amenities_parsed']:
    for amenity in amenities:
        if amenity not in amenities_list:
            amenities_list.append(amenity)

In [30]:
listings.head()

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,amenities_parsed,content
0,35797,https://www.airbnb.com/rooms/35797,20250625031918,2025-06-26,city scrape,Villa Dante,"Dentro de Villa un estudio de arte con futon, ...","Santa Fe Shopping Mall, Interlomas Park and th...",https://a0.muscache.com/pictures/f395ab78-1185...,153786,https://www.airbnb.com/users/show/153786,Dici,2010-06-28,"Mexico City, Mexico","Master in visual arts, film photography & Mark...",NaN,NaN,NaN,f,https://a0.muscache.com/im/pictures/user/00de1...,https://a0.muscache.com/im/pictures/user/00de1...,NaN,1.00,1.00,"['email', 'phone', 'work_email']",t,t,"Mexico City, D.f., Mexico",Cuajimalpa de Morelos,NaN,19.38,-99.27,Entire villa,Entire home/apt,2,1.00,1 bath,1.00,1.00,"[""Kitchen"", ""Resort access"", ""Hot water"", ""Cou...","$3,799.00",1,7,1.00,1.00,7.00,7.00,1.00,7.00,NaN,t,29,59,89,364,2025-06-26,0,0,0,188,0,0,0.00,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,f,1,1,0,0,NaN,"[Kitchen, Resort access, Hot water, Courtyard ...",Kitchen Resort access Hot water Courtyard view...
1,44616,https://www.airbnb.com/rooms/44616,20250625031918,2025-07-01,city scrape,Condesa Haus,A new concept of hosting in mexico through a b...,NaN,https://a0.muscache.com/pictures/251410/ec75fe...,196253,https://www.airbnb.com/users/show/196253,Fernando,2010-08-09,"Mexico City, Mexico",Condesa Haus Rentals offers independent stud...,within an hour,100%,91%,f,https://a0.muscache.com/im/users/196253/profil...,https://a0.muscache.com/im/users/196253/profil...,Condesa,13.00,13.00,"['email', 'phone', 'work_email']",t,t,NaN,Cuauhtémoc,NaN,19.41,-99.18,Entire home,Entire home/apt,14,5.50,5.5 baths,5.00,8.00,"[""Free street parking"", ""Free parking on premi...","$18,000.00",1,180,1.00,1.00,180.00,180.00,1.00,180.00,NaN,t,29,59,89,360,2025-07-01,65,1,0,179,0,6,108000.00,2011-11-09,2025-01-01,4.59,4.56,4.70,4.87,4.78,4.98,4.47,NaN,f,9,4,2,0,0.39,"[Free street parking, Free parking on premises...",Free street parking Free parking on premises E...
2,56074,https://www.airbnb.com/rooms/56074,20250625031918,2025-07-01,city scrape,Great space in historical San Rafael,This great apartment is located in one of the ...,Very traditional neighborhood with all service...,https://a0.muscache.com/pictures/3005118/60dac...,265650,https://www.airbnb.com/users/show/265650,Maris,2010-10-19,"Mexico City, Mexico",I am a University Professor now retired after ...,within a few hours,100%,100%,f,https://a0.muscache.com/im/users/265650/profil...,https://a0.muscache.com/im/users/265650/profil...,San Rafael,1.00,5.00,"['email', 'phone']",t,t,"Mexico City, DF, Mexico",Cuauhtémoc,NaN,19.44

In [40]:
for i in listings.index:
    for val in listings.loc[i, 'amenities_parsed']:
        if 'Hot tub' == val:
            listings.at[i, 'Hot tub'] = 1

In [41]:
listings[listings['Hot tub']==1]

,id,listing_url,scrape_id,last_scraped,source,name,description,neighborhood_overview,picture_url,host_id,host_url,host_name,host_since,host_location,host_about,host_response_time,host_response_rate,host_acceptance_rate,host_is_superhost,host_thumbnail_url,host_picture_url,host_neighbourhood,host_listings_count,host_total_listings_count,host_verifications,host_has_profile_pic,host_identity_verified,neighbourhood,neighbourhood_cleansed,neighbourhood_group_cleansed,latitude,longitude,property_type,room_type,accommodates,bathrooms,bathrooms_text,bedrooms,beds,amenities,price,minimum_nights,maximum_nights,minimum_minimum_nights,maximum_minimum_nights,minimum_maximum_nights,maximum_maximum_nights,minimum_nights_avg_ntm,maximum_nights_avg_ntm,calendar_updated,has_availability,availability_30,availability_60,availability_90,availability_365,calendar_last_scraped,number_of_reviews,number_of_reviews_ltm,number_of_reviews_l30d,availability_eoy,number_of_reviews_ly,estimated_occupancy_l365d,estimated_revenue_l365d,first_review,last_review,review_scores_rating,review_scores_accuracy,review_scores_cleanliness,review_scores_checkin,review_scores_communication,review_scores_location,review_scores_value,license,instant_bookable,calculated_host_listings_count,calculated_host_listings_count_entire_homes,calculated_host_listings_count_private_rooms,calculated_host_listings_count_shared_rooms,reviews_per_month,amenities_parsed,content,has_pool,Hot tub
19,187745,https://www.airbnb.com/rooms/187745,20250625031918,2025-07-01,previous scrape,Extraordinarily Spacious Apt. in Condesa,HE NEIGHBORHOOD<br /><br />Condesa is the plac...,Condesa is the place to be in Mexico City. Oft...,https://a0.muscache.com/pictures/c4d9a98f-119b...,899360,https://www.airbnb.com/users/show/899360,Julian,2011-07-31,NaN,Welcome home!,NaN,NaN,100%,t,https://a0.muscache.com/im/pictures/user/4c97f...,https://a0.muscache.com/im/pictures/user/4c97f...,Condesa,6.00,8.00,"['email', 'phone']",t,t,"Mexico City, Federal District, Mexico",Cuauhtémoc,NaN,19.41,-99.18,Entire loft,Entire home/apt,3,NaN,2 baths,1.00,NaN,"[""Free street parking"", ""Hot water"", ""TV with ...",NaN,14,365,14.00,14.00,365.00,365.00,14.00,365.00,NaN,t,0,0,0,0,2025-07-01,24,0,0,0,0,0,NaN,2013-11-10,2018-01-03,4.67,4.58,4.58,4.83,4.79,5.00,4.63,NaN,f,6,6,0,0,0.17,"[Free street parking, Hot water, TV with stand...",Free street parking Hot water TV with standard...,NaN,1.00
28,276504,https://www.airbnb.com/rooms/276504,20250625031918,2025-07-01,previous scrape,High End Condo with golf package,NaN,NaN,https://a0.muscache.com/pictures/2802432/4be14...,1444589,https://www.airbnb.com/users/show/1444589,Michael,2011-11-26,NaN,NaN,NaN,NaN,NaN,f,https://a0.muscache.com/im/users/1444589/profi...,https://a0.muscache.com/im/users/1444589/profi...,NaN,1.00,1.00,"['email', 'phone']",t,f,NaN,Iztacalco,NaN,19.38,-99.13,Entire rental unit,Entire home/apt,2,NaN,1 bath,NaN,NaN,"[""Free parking on premises"", ""Washer"", ""Dryer""...",NaN,7,365,7.00,7.00,365.00,365.00,7.00,365.00,NaN,NaN,0,0,0,0,2025-07-01,0,0,0,0,0,0,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,f,1,1,0,0,NaN,"[Free parking on premises, Washer, Dryer, Air ...",Free parking on premises Washer Dryer Air cond...,1.00,1.00
39,298873,https://www.airbnb.com/rooms/298873,20250625031918,2025-07-01,previous scrape,"The ""home"" feeling in Mexico City!",NaN,NaN,https://a0.muscache.com/pictures/9032323/9bdb5...,1539548,https://www.airbnb.com/users/show/1539548,Paulina,2011-12-27,Switzerland,"Hi, I am a photographer/journalist, who loves ...",NaN,NaN,NaN,f,https://a0.muscache.com/im/users/1539548/profi...,https://a0.muscache.com/im/users/1539548/profi...,Centro Histórico,1.00,3.00,"['email', 'phone']",t,f,NaN,Cuauhtémoc,NaN,19.43,-99.14,Entire rental unit,Entire home/apt,4,NaN,2 baths,2.00,NaN,"[""Free parking on premises"", ""Dryer"", ""Hot wat...",NaN,3,180,3.00,3.00,180.00,180.00,3.00,180.00,NaN,t,0,0,0,0,2025-07-01,119,0,0,0,0,0,NaN,2012-04-19,2019-11-03,4.78,4.87,4.88,4.8